# Gaussian Molecular Shape - Student Notebook

This notebook teaches how to **represent a molecule's shape with Gaussians**, **compare two shapes**, and **align them**. Run the cells top to bottom, read the notes, and try the exercises at the end.

Companion reading: `GaussianShape_tutorial.md`.

**Prerequisites:** `numpy`, `scipy`, `matplotlib`, `rdkit`.

## 0. Setup

We add the project root to `sys.path` so the `core` and `visualize` packages import. Run this notebook from the project root (`comp_dynamics/`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem

from core.gaussian_shape import (
    GaussianShape, align_shapes_bfgs, apply_transform_to_mol, VDW_RADII,
)
from visualize.plot_shape import plot_density_slice, plot_shape_overlay
print('imports OK')

## 1. One atom = one Gaussian

Each atom `i` at position `r_i` becomes an isotropic Gaussian

$$ g_i(\mathbf{r}) = \exp\left(-\frac{\lVert \mathbf{r}-\mathbf{r}_i\rVert^2}{2\sigma_i^2}\right) $$

The width `sigma` sets how 'fat' the atom is. Below we plot it in 1D for three widths.

In [ ]:
x = np.linspace(-5, 5, 400)
for sigma in (0.7, 1.2, 2.0):
    plt.plot(x, np.exp(-x**2 / (2*sigma**2)), label=f'sigma={sigma}')
plt.title('single-atom Gaussian'); plt.xlabel('r - r_i'); plt.ylabel('g(r)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 2. A molecule = a sum of Gaussians

$$ \rho(\mathbf{r}) = \sum_i g_i(\mathbf{r}) $$

Let's build a real 3D molecule with RDKit and wrap it in a `GaussianShape`. The `sigma` of each atom is `radius_scale * R_vdW`.

In [ ]:
def make_mol_3d(smiles, seed=0):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    p = AllChem.ETKDGv3(); p.randomSeed = seed
    AllChem.EmbedMolecule(mol, p)
    AllChem.MMFFOptimizeMolecule(mol)
    return mol

benzene = make_mol_3d('c1ccccc1')
shape = GaussianShape.from_rdkit_mol(benzene, radius_scale=0.8)
print('atoms:', shape.n_atoms)
print('centroid:', np.round(shape.centroid(), 3))
print('self-overlap S_AA:', round(shape.self_overlap(), 3))

## 3. Look at the density

`plot_density_slice` evaluates `rho(r)` on a plane and draws it as a heatmap.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_density_slice(shape, axis='z', ax=ax, title='benzene density (z-slice)')
plt.show()

## 4. Overlap and Tanimoto similarity

Pairwise Gaussian overlap has a closed form:

$$ S_{ab} = \left(\frac{2\pi\sigma_a^2\sigma_b^2}{\sigma_a^2+\sigma_b^2}\right)^{3/2} \exp\left(-\frac{\lVert a-b\rVert^2}{2(\sigma_a^2+\sigma_b^2)}\right) $$

Total overlap sums over all atom pairs, and similarity is

$$ T = \frac{S_{AB}}{S_{AA}+S_{BB}-S_{AB}} \in [0, 1] $$

In [ ]:
cyclohexane = make_mol_3d('C1CCCCC1')
shape_b = GaussianShape.from_rdkit_mol(cyclohexane, radius_scale=0.8)

print('S_AB    :', round(shape.overlap(shape_b), 3))
print('Tanimoto:', round(shape.tanimoto(shape_b), 3))
print('note: this is BEFORE alignment, so it depends on the RDKit poses.')

## 5. Alignment: compare in a common frame

Tanimoto depends on relative pose. We displace a copy of a molecule far away, then recover the transform by **maximizing overlap** with BFGS (analytical gradient, several random restarts).

Because self-overlaps are constant under rigid motion, maximizing `S_AB` also maximizes `T`.

In [ ]:
from scipy.spatial.transform import Rotation

ref_mol = make_mol_3d('CC(C)Cc1ccc(cc1)C(C)C(=O)O', seed=1)  # ibuprofen
ref_shape = GaussianShape.from_rdkit_mol(ref_mol, radius_scale=0.8)

mobile_mol = Chem.Mol(ref_mol)
R_rand = Rotation.random(random_state=7).as_matrix()
t_rand = np.array([30.0, -20.0, 25.0])
apply_transform_to_mol(mobile_mol, R_rand, t_rand)
mobile_shape = GaussianShape.from_rdkit_mol(mobile_mol, radius_scale=0.8)

print('Tanimoto before:', round(ref_shape.tanimoto(mobile_shape), 4))
result = align_shapes_bfgs(mobile_shape, ref_shape, n_starts=16)
print('Tanimoto after :', round(result.tanimoto, 4))
print('4x4 transform matrix:')
print(np.array2string(result.matrix, precision=3, suppress_small=True))

## 6. Apply the transform and export for PyMOL

The returned `rotation` / `translation` act on the ORIGINAL mobile coordinates, so we can move the real RDKit molecule and write SDF files.

In [ ]:
os.makedirs('examples/output', exist_ok=True)
aligned_mol = Chem.Mol(mobile_mol)
apply_transform_to_mol(aligned_mol, result.rotation, result.translation)

for mol, fname in [(ref_mol, 'nb_ref.sdf'), (mobile_mol, 'nb_start.sdf'), (aligned_mol, 'nb_aligned.sdf')]:
    w = Chem.SDWriter(f'examples/output/{fname}'); w.write(mol); w.close()

# check: aligned molecule reproduces the aligned shape
chk = GaussianShape.from_rdkit_mol(aligned_mol, radius_scale=0.8)
print('max coord deviation:', np.abs(chk.centers - result.aligned.centers).max())
print('wrote nb_ref.sdf, nb_start.sdf, nb_aligned.sdf to examples/output/')

## Exercises

Fill in the `...` and run. Compare your results with a neighbor!

**E1.** How does `radius_scale` affect similarity? Compute the benzene-cyclohexane Tanimoto for scales 0.5, 0.8, 1.0.

In [ ]:
for scale in (0.5, 0.8, 1.0):
    sa = GaussianShape.from_rdkit_mol(benzene, radius_scale=scale)
    sb = GaussianShape.from_rdkit_mol(cyclohexane, radius_scale=scale)
    # TODO: print scale and sa.tanimoto(sb)
    ...


**E2.** Heavy-atom-only shapes. Rebuild `ref_shape` with `include_hydrogens=False` and compare the self-overlap and atom count to the all-atom version.

In [ ]:
# TODO: build heavy-atom-only shape and compare
heavy = GaussianShape.from_rdkit_mol(ref_mol, radius_scale=0.8, include_hydrogens=...)
...


**E3.** Robustness of alignment. Re-run `align_shapes_bfgs` with `n_starts = 1, 2, 4, 8`. What is the smallest number of restarts that still recovers Tanimoto ~ 1.0 for the displaced ibuprofen?

In [ ]:
for n in (1, 2, 4, 8):
    # TODO: align and print n, result.tanimoto
    ...


**E4 (challenge).** Compare two DIFFERENT conformers of the same flexible molecule (e.g. `make_mol_3d('CCCCCCCC', seed=1)` vs `seed=5`). Align them and inspect the best Tanimoto - it should be < 1.0. Visualize with `plot_shape_overlay`.

In [ ]:
# TODO: build two conformers, align, overlay-plot
...
